# Advanced Tree Decompositions

The **tree algorithms** notebook traversed trees directly — LCA, subtree sums, diameter. That works until queries and updates **interleave** on a tree with no natural array layout. The move here is to *flatten the tree into an array* so that **path** and **subtree** questions become **range** questions a Fenwick or segment tree can answer in O(log n). Two decompositions do the heavy lifting:

- **Centroid decomposition** — recursively peel off **centroids** to build a balanced auxiliary *centroid tree* of depth O(log n); every path in the original tree passes through a small set of centroids, turning "all pairs / paths through a point" problems into per-centroid bookkeeping.
- **Heavy-light decomposition (HLD)** — split the tree into **heavy chains**, lay each chain contiguously in an array, and answer any root-to-node path with O(log n) chain segments — each a range query on a **segment tree** (builds on the **fenwick & segment trees** notebook).

Every claim below is checked against brute-force tree traversal over many seeded-random trees.

**Contents**

1. **The idea** — why decompose
2. **Centroid decomposition** — the centroid tree + counting pairs within distance K
3. **Heavy-light decomposition** — heavy chains + path-sum / path-max queries
4. **When to use what**


## 1. The idea — why decompose

On an array, a segment tree makes range queries cheap. A tree has no such linear order — the path between two nodes can zig-zag through O(n) edges, and a subtree is scattered. **Decomposition imposes a structure** so those queries collapse into ranges or into a logarithmic walk:

| Scheme | What it builds | Turns into | Good for |
|---|---|---|---|
| **Euler tour** (in the **tree algorithms** notebook) | enter/exit times | a **contiguous range** per subtree | subtree aggregate + point update |
| **Centroid decomposition** | a balanced *centroid tree*, depth O(log n) | "paths through a centroid" buckets | count / aggregate over **all paths** by length |
| **Heavy-light decomposition** | O(n) heavy chains, each contiguous in an array | O(log n) range queries per **path** | path-sum / path-max with updates |

The shared trick: pay an O(n) or O(n log n) preprocessing cost once, then ride a logarithmic structure for every query. We build both from scratch and prove them against brute force.

In [1]:
import random
from collections import deque

def random_tree(n, rng):
    """Random labelled tree: node v (>=1) attaches to a uniformly random earlier node."""
    adj = [[] for _ in range(n)]
    for v in range(1, n):
        u = rng.randint(0, v - 1)
        adj[u].append(v); adj[v].append(u)
    return adj

def bfs_dist(adj, src):                       # brute-force single-source distances
    dist = [-1] * len(adj); dist[src] = 0; q = deque([src])
    while q:
        u = q.popleft()
        for w in adj[u]:
            if dist[w] == -1:
                dist[w] = dist[u] + 1; q.append(w)
    return dist

rng = random.Random(0)
adj = random_tree(7, rng)
print("a random 7-node tree (adjacency):")
for u, nbrs in enumerate(adj):
    print(f"  {u}: {nbrs}")

a random 7-node tree (adjacency):
  0: [1, 3]
  1: [0, 2]
  2: [1, 4]
  3: [0, 6]
  4: [2, 5]
  5: [4]
  6: [3]


## 2. Centroid decomposition

A **centroid** of a tree is a node whose removal leaves every remaining piece with **at most ⌊n/2⌋** nodes — the most balanced possible split. Centroid decomposition recurses on that: find the centroid of the whole tree, remove it, then decompose each leftover piece, recording each centroid's parent as the centroid that removed *its* component. The result is the **centroid tree**.

Because every split at least **halves** the component size, the centroid tree has depth **O(log n)** — even if the original tree is a degenerate path. That bound is the whole point: any path in the original tree, between nodes `u` and `v`, passes through exactly one node that is an **ancestor of both in the centroid tree** (their meeting centroid), so problems about *all paths* reduce to "for each centroid, look only at paths that pass through it."

We find a centroid by computing subtree sizes once, then walking toward the heaviest child until no child exceeds half the total — all iterative, so deep trees don't blow the recursion limit.

In [2]:
def find_centroid(adj, removed, subtree, entry):
    """Subtree sizes within the live component rooted at `entry`, then walk to its centroid."""
    order, par, stack = [], {entry: -1}, [entry]
    while stack:                              # iterative DFS to list the component
        u = stack.pop(); order.append(u)
        for w in adj[u]:
            if w != par[u] and not removed[w]:
                par[w] = u; stack.append(w)
    for u in reversed(order):                 # children finish before parents -> sizes
        subtree[u] = 1
        for w in adj[u]:
            if w != par[u] and not removed[w]:
                subtree[u] += subtree[w]
    total = len(order)
    u, p = entry, -1
    while True:                               # move toward any child holding > half
        nxt = next((w for w in adj[u]
                    if w != p and not removed[w] and subtree[w] > total // 2), -1)
        if nxt == -1:
            return total, u
        subtree[u] = total - subtree[nxt]     # re-root the size as we step in
        p, u = u, nxt

def centroid_tree(adj):
    """Return cpar[c] = the centroid that removed c's component (-1 for the root)."""
    n = len(adj)
    removed = [False] * n; subtree = [0] * n; cpar = [-1] * n
    work = [(0, -1)]                          # (entry node, parent centroid)
    while work:
        entry, par = work.pop()
        _, c = find_centroid(adj, removed, subtree, entry)
        cpar[c] = par; removed[c] = True
        for w in adj[c]:
            if not removed[w]:
                work.append((w, c))
    return cpar

# depth of the centroid tree vs the original tree's (much larger) height
import math
path = [[] for _ in range(16)]               # a degenerate path 0-1-2-...-15
for i in range(15):
    path[i].append(i + 1); path[i + 1].append(i)
cpar = centroid_tree(path)
def ct_depth(cpar):
    depth = [-1] * len(cpar)
    def d(x):
        if x == -1: return -1
        if depth[x] < 0: depth[x] = d(cpar[x]) + 1
        return depth[x]
    return max(d(x) for x in range(len(cpar)))
print(f"path of 16 nodes: original height 15, centroid-tree depth {ct_depth(cpar)} "
      f"(<= ceil(log2 16) = {math.ceil(math.log2(16))})")

path of 16 nodes: original height 15, centroid-tree depth 4 (<= ceil(log2 16) = 4)


In [3]:
# PROOF 1 - the halving invariant: removing a centroid leaves every piece <= floor(T/2)
rng = random.Random(2024)
for _ in range(500):
    n = rng.randint(1, 50)
    adj = random_tree(n, rng)
    removed = [False] * n; subtree = [0] * n
    work = [0]
    while work:
        total, c = find_centroid(adj, removed, subtree, work.pop())
        removed[c] = True
        for w in adj[c]:
            if not removed[w]:                # measure each leftover piece by BFS
                sz, q, seen = 0, deque([w]), {w, c}
                while q:
                    x = q.popleft(); sz += 1
                    for y in adj[x]:
                        if y not in seen and not removed[y]:
                            seen.add(y); q.append(y)
                assert sz <= total // 2
                work.append(w)
print("halving invariant holds: every piece <= floor(T/2)  [500 random trees] OK")

# PROOF 2 - the centroid tree is valid: every node once, exactly one root, depth <= ceil(log2 n)+1
rng = random.Random(42)
for _ in range(400):
    n = rng.randint(1, 40)
    cp = centroid_tree(random_tree(n, rng))
    assert sorted(x for x in range(n)) == list(range(n))         # all nodes present
    assert sum(1 for x in range(n) if cp[x] == -1) == 1          # single root
    bound = math.ceil(math.log2(n)) if n > 1 else 0
    assert ct_depth(cp) <= bound + 1
print("centroid tree valid + depth <= ceil(log2 n)+1            [400 random trees] OK")

halving invariant holds: every piece <= floor(T/2)  [500 random trees] OK
centroid tree valid + depth <= ceil(log2 n)+1            [400 random trees] OK


### Application — count pairs of nodes at distance ≤ K

The payoff: counting how many of the n(n−1)/2 node pairs lie within distance **K** of each other. Brute force runs a BFS from every node — O(n²). Centroid decomposition does it in **O(n log² n)**: at each centroid `c`, every path *through* `c` is a distance-from-`c` on one side plus a distance-from-`c` on the other. Gather all distances from `c`, count pairs whose distances sum to ≤ K with a two-pointer sweep, then **subtract** the pairs that came from the *same* branch (those paths don't actually pass through `c` — they'd be counted again, correctly, deeper in the recursion).

In [4]:
def count_pairs_within_k(adj, K):
    n = len(adj)
    removed = [False] * n; subtree = [0] * n; total_pairs = 0

    def branch_dists(start, c):               # distances from c through neighbor `start`
        res, stack = [], [(start, c, 1)]
        while stack:
            u, p, d = stack.pop(); res.append(d)
            for w in adj[u]:
                if w != p and not removed[w]:
                    stack.append((w, u, d + 1))
        return res

    def count_le(arr):                        # pairs (i<j) with arr[i]+arr[j] <= K
        arr.sort(); i, j, c = 0, len(arr) - 1, 0
        while i < j:
            if arr[i] + arr[j] <= K: c += j - i; i += 1
            else: j -= 1
        return c

    work = [0]
    while work:
        _, c = find_centroid(adj, removed, subtree, work.pop())
        full = [0]                            # c itself, at distance 0
        for w in adj[c]:
            if not removed[w]:
                b = branch_dists(w, c)
                full += b
                total_pairs -= count_le(b)    # same-branch pairs don't pass through c
        total_pairs += count_le(full)
        removed[c] = True
        for w in adj[c]:
            if not removed[w]:
                work.append(w)
    return total_pairs

# brute force: BFS from every node, tally pairs within K
def brute_pairs(adj, K):
    n, cnt = len(adj), 0
    for s in range(n):
        d = bfs_dist(adj, s)
        cnt += sum(1 for t in range(s + 1, n) if d[t] <= K)
    return cnt

demo = random_tree(9, random.Random(1))
for K in (1, 2, 3):
    print(f"pairs within distance {K}: centroid={count_pairs_within_k(demo, K)}  "
          f"brute={brute_pairs(demo, K)}")

# PROOF 3 - matches brute force across many trees and K values
rng = random.Random(7)
for _ in range(500):
    n = rng.randint(1, 35)
    adj = random_tree(n, rng)
    for K in range(0, 8):
        assert count_pairs_within_k(adj, K) == brute_pairs(adj, K)
print("count-pairs-within-K matches brute force  [500 trees x 8 values of K] OK")

pairs within distance 1: centroid=8  brute=8
pairs within distance 2: centroid=19  brute=19
pairs within distance 3: centroid=27  brute=27


count-pairs-within-K matches brute force  [500 trees x 8 values of K] OK


## 3. Heavy-light decomposition

Centroid decomposition shines on *path-counting*; for **path queries with updates** — "sum (or max) of values on the path from u to v, with point updates" — the tool is **heavy-light decomposition (HLD)**.

Label each edge from a node to its child **heavy** if that child holds the largest subtree (ties broken arbitrarily), **light** otherwise. Following heavy edges carves the tree into disjoint **heavy chains**. The key fact: walking from any node to the root crosses at most **O(log n) light edges** — because each light edge drops you into a subtree less than half the size — so any root-to-node path spans **O(log n) chains**.

Lay each chain **contiguously** in a base array (DFS so a chain occupies a consecutive block) and build a **segment tree** over it (the **fenwick & segment trees** notebook). A path `u → v` splits at their LCA into O(log n) chain segments; each is one **range query**, giving **O(log² n)** per path query and **O(log n)** per point update. We compute parents, depths, subtree sizes, and heavy children all iteratively.

In [5]:
class SegTree:
    """Iterative point-update / range-query segment tree over any associative combine."""
    def __init__(self, data, combine, identity):
        self.n, self.combine, self.identity = len(data), combine, identity
        self.t = [identity] * (2 * self.n)
        for i, x in enumerate(data): self.t[self.n + i] = x
        for i in range(self.n - 1, 0, -1):
            self.t[i] = combine(self.t[2 * i], self.t[2 * i + 1])
    def update(self, i, v):
        i += self.n; self.t[i] = v; i //= 2
        while i:
            self.t[i] = self.combine(self.t[2 * i], self.t[2 * i + 1]); i //= 2
    def query(self, l, r):                    # half-open [l, r)
        res = self.identity; l += self.n; r += self.n
        while l < r:
            if l & 1: res = self.combine(res, self.t[l]); l += 1
            if r & 1: r -= 1; res = self.combine(res, self.t[r])
            l //= 2; r //= 2
        return res

In [6]:
class HLD:
    def __init__(self, adj, values, root=0):
        n = len(adj)
        self.adj, self.depth = adj, [0] * n
        self.parent = [-1] * n; self.heavy = [-1] * n
        self.head = [0] * n; self.pos = [0] * n
        size = [1] * n
        # 1) iterative DFS for parent + depth, recording visit order
        order, st = [], [root]
        while st:
            u = st.pop(); order.append(u)
            for w in adj[u]:
                if w != self.parent[u]:
                    self.parent[w] = u; self.depth[w] = self.depth[u] + 1; st.append(w)
        # 2) subtree sizes + heavy child (children precede parents in reversed order)
        for u in reversed(order):
            best, bsz = -1, 0
            for w in adj[u]:
                if w != self.parent[u]:
                    size[u] += size[w]
                    if size[w] > bsz: bsz, best = size[w], w
            self.heavy[u] = best
        # 3) lay chains contiguously: each chain start descends its heavy path
        cur, starts = 0, [root]
        while starts:
            v = starts.pop(); s = v
            while v != -1:
                self.head[v] = s; self.pos[v] = cur; cur += 1
                for w in adj[v]:              # light children spawn new chains
                    if w != self.parent[v] and w != self.heavy[v]:
                        starts.append(w)
                v = self.heavy[v]
        base = [0] * n
        for u in range(n): base[self.pos[u]] = values[u]
        self.values = values[:]
        self.sum = SegTree(base, lambda a, b: a + b, 0)
        self.mx  = SegTree(base, lambda a, b: a if a > b else b, float("-inf"))

    def update(self, u, val):
        self.values[u] = val
        self.sum.update(self.pos[u], val); self.mx.update(self.pos[u], val)

    def _segments(self, u, v):                # half-open base-array ranges covering path u..v
        out = []
        while self.head[u] != self.head[v]:
            if self.depth[self.head[u]] < self.depth[self.head[v]]: u, v = v, u
            out.append((self.pos[self.head[u]], self.pos[u] + 1))
            u = self.parent[self.head[u]]
        if self.depth[u] > self.depth[v]: u, v = v, u
        out.append((self.pos[u], self.pos[v] + 1))
        return out

    def path_sum(self, u, v):
        return sum(self.sum.query(l, r) for l, r in self._segments(u, v))
    def path_max(self, u, v):
        return max(self.mx.query(l, r) for l, r in self._segments(u, v))

# small demo on a fixed tree
adj = [[1, 2], [0, 3, 4], [0], [1], [1, 5], [4]]      # 0-1-{3,4-5}, 0-2
vals = [3, 1, 4, 1, 5, 9]
hld = HLD(adj, vals, root=0)
print("values        :", vals)
print("path_sum(3,5) :", hld.path_sum(3, 5), "(nodes 3,1,4,5 -> 1+1+5+9 = 16)")
print("path_max(3,2) :", hld.path_max(3, 2), "(nodes 3,1,0,2 -> max 4)")
hld.update(4, 0)
print("after a[4]=0, path_max(3,5):", hld.path_max(3, 5), "(nodes 3,1,4,5 -> max 9)")

values        : [3, 1, 4, 1, 5, 9]
path_sum(3,5) : 16 (nodes 3,1,4,5 -> 1+1+5+9 = 16)
path_max(3,2) : 4 (nodes 3,1,0,2 -> max 4)
after a[4]=0, path_max(3,5): 9 (nodes 3,1,4,5 -> max 9)


In [7]:
# PROOF 4 - path_sum / path_max with random updates vs brute-force path walking
def tree_path(adj, u, v):                     # nodes on the u..v path, by BFS + backtrack
    par = {u: u}; q = deque([u])
    while q:
        x = q.popleft()
        if x == v: break
        for w in adj[x]:
            if w not in par: par[w] = x; q.append(w)
    path = [v]
    while path[-1] != u: path.append(par[path[-1]])
    return path

rng = random.Random(123)
for _ in range(400):
    n = rng.randint(1, 40)
    adj = random_tree(n, rng)
    vals = [rng.randint(-50, 50) for _ in range(n)]
    hld = HLD(adj, vals, root=0)
    for _ in range(15):
        if rng.random() < 0.4:                # a point update
            node, nv = rng.randint(0, n - 1), rng.randint(-50, 50)
            hld.update(node, nv); vals[node] = nv
        else:                                 # a path query
            u, v = rng.randint(0, n - 1), rng.randint(0, n - 1)
            nodes = tree_path(adj, u, v)
            assert hld.path_sum(u, v) == sum(vals[x] for x in nodes)
            assert hld.path_max(u, v) == max(vals[x] for x in nodes)
print("HLD path_sum / path_max match brute force under random updates  [400 trees] OK")

# PROOF 5 - any path decomposes into O(log n) chain segments
import math
worst = 0
rng = random.Random(99)
for _ in range(200):
    n = rng.randint(2, 64)
    adj = random_tree(n, rng); hld = HLD(adj, [0] * n, 0)
    for _ in range(20):
        u, v = rng.randint(0, n - 1), rng.randint(0, n - 1)
        k = len(hld._segments(u, v)); worst = max(worst, k)
        assert k <= 2 * (math.floor(math.log2(n)) + 1)
print(f"every path uses O(log n) chain segments (worst seen: {worst})         [200 trees] OK")

HLD path_sum / path_max match brute force under random updates  [400 trees] OK
every path uses O(log n) chain segments (worst seen: 6)         [200 trees] OK


## 4. When to use what

| You need… | Reach for | Cost |
|---|---|---|
| **Subtree** aggregate + point update | **Euler tour** + Fenwick/segment tree (**tree algorithms** notebook) | O(log n) |
| Count / aggregate over **all paths** by length or property | **centroid decomposition** | O(n log n) build, O(n log² n) typical |
| **Path** sum / max / min with point updates | **heavy-light decomposition** + segment tree | O(log² n) per query, O(log n) update |
| Path queries **+ range updates** on the path | HLD + **lazy** segment tree (**fenwick & segment trees** notebook) | O(log² n) |
| Offline path/subtree queries, no updates | Euler tour + sort, or small-to-large merging | O(n log n) |

**Choosing between the two stars:** centroid decomposition reorganizes the tree so that *every path is owned by exactly one centroid* — ideal for **counting** problems ("how many pairs within distance K", "paths summing to S"). HLD instead makes a *single* path cheap to query and update repeatedly — ideal for **online path queries**. Euler tour is the lightest option and the right default whenever your queries are about **subtrees** rather than arbitrary paths.

| Scheme | Build | Per query | Handles updates? | Best at |
|---|:---:|:---:|:---:|---|
| Euler tour + segtree | O(n) | O(log n) | yes (point) | subtree aggregates |
| Centroid decomposition | O(n log n) | O(log² n) | rebuild-ish | all-paths counting |
| Heavy-light decomposition | O(n) | O(log² n) | yes (point/range) | online path queries |

All three are the same bargain: spend a near-linear preprocessing pass to **flatten an awkward tree query into a logarithmic array query** — the recurring theme from **fenwick & segment trees** carried onto trees.